In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from numpy.polynomial.polynomial import polymul
from ipywidgets import RadioButtons, FloatSlider, HTML, HBox, VBox, Layout
from IPython.display import display

# ============================================================
# DIGITAL FILTER TRANSFORMATION EXERCISE
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.8})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.tr-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.tr-header{
    display:flex;
    justify-content:space-between;
    align-items:center;
    background:#263238;
    color:white;
    padding:10px 14px;
    border-radius:8px 8px 0 0;
}

.tr-header-title{
    font-size:19px;
    font-weight:bold;
}

.tr-badge{
    background:white;
    color:#263238;
    border-radius:16px;
    padding:4px 13px;
    font-size:15px;
    font-weight:bold;
}

.tr-info{
    border:1px solid #b0bec5;
    border-top:none;
    border-radius:0 0 8px 8px;
    background:#f7f9fa;
    padding:8px 12px;
    margin-bottom:7px;
    font-size:13.5px;
    line-height:1.45;
}

.tr-result{
    width:900px;
    box-sizing:border-box;
    background:#fffdf3;
    border:1px solid #d7c676;
    border-radius:7px;
    padding:8px 12px;
    margin-bottom:6px;
    font-size:13.5px;
    line-height:1.45;
}

.tr-result-title{
    font-size:14.5px;
    font-weight:bold;
    color:#5d4037;
    margin-bottom:5px;
}

.tr-equation{
    text-align:center;
    font-family:serif;
    font-size:16px;
    margin:5px 0;
}

.horizontal-radio .widget-radio-box{
    display:flex !important;
    flex-direction:row !important;
    flex-wrap:nowrap !important;
    gap:25px !important;
}

.horizontal-radio .widget-radio-box label{
    margin:0 !important;
    white-space:nowrap !important;
    font-weight:bold;
}

.horizontal-radio > label{
    display:none !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# HEADER
# ============================================================

header_html = HTML("""
<div class="tr-root">

<div class="tr-header">
    <div class="tr-header-title">Digital Filter Transformation Exercise</div>
    <div class="tr-badge">LP Prototype</div>
</div>

<div class="tr-info">
Select the desired digital filter transformation. The controls are automatically adapted to the selected case, and the corresponding transformed amplitude response is calculated from the prototype low-pass filter.
</div>

</div>
""")

# ============================================================
# PROTOTYPE FILTER
# ============================================================

Omega_p = 0.15*np.pi
Omega_s = 0.35*np.pi

b = np.array([0.008616,0.025848,0.025848,0.008616],dtype=float)
a = np.array([1.000000,-2.064414,1.519112,-0.385767],dtype=float)

# ============================================================
# RADIO BUTTONS
# ============================================================

transformation_radio = RadioButtons(options=['LP → LP','LP → HP','LP → BP','LP → BS'],value='LP → LP',description='')

transformation_radio.add_class('horizontal-radio')

radio_box = HBox([transformation_radio],layout=Layout(width=CONTENT_WIDTH,height='54px',border='1px solid #b0bec5',padding='8px 12px 12px 12px',margin='0 0 6px 0'))

# ============================================================
# CONTROLS
# ============================================================

wp_slider = FloatSlider(value=0.65,min=0.20,max=0.85,step=0.01,description='ωp/π:',continuous_update=True,readout_format='.2f',style={'description_width':'42px'},layout=Layout(width='420px'))

wp1_slider = FloatSlider(value=0.45,min=0.10,max=0.75,step=0.01,description='ωp1/π:',continuous_update=True,readout_format='.2f',style={'description_width':'52px'},layout=Layout(width='420px'))

wp2_slider = FloatSlider(value=0.75,min=0.25,max=0.95,step=0.01,description='ωp2/π:',continuous_update=True,readout_format='.2f',style={'description_width':'52px'},layout=Layout(width='420px'))

single_frequency_box = HBox([wp_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b0bec5',padding='7px 12px',margin='0 0 2px 0'))

band_frequency_box = HBox([wp1_slider,wp2_slider],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b0bec5',padding='7px 12px',margin='0 0 2px 0'))

# ============================================================
# RESULT PANEL
# ============================================================

result_html = HTML(layout=Layout(width=CONTENT_WIDTH))

# ============================================================
# POLYNOMIAL UTILITIES
# ============================================================

def polynomial_power(p,n):

    result = np.array([1.0])

    for _ in range(n):

        result = polymul(result,p)

    return result

def substitute_rational_polynomial(c,P,Q,N):

    degree = N*(len(P)-1)

    result = np.zeros(degree+1)

    for k,ck in enumerate(c):

        if k > N:

            break

        term = ck*polymul(polynomial_power(P,k),polynomial_power(Q,N-k))

        result[:len(term)] += term

    return result

def clean_coefficients(c,tol=1e-12):

    result = c.copy()

    result[np.abs(result) < tol] = 0.0

    return result

# ============================================================
# LP -> LP
# ============================================================

def transform_lp_to_lp(b,a,Omega_p,omega_p):

    alpha = np.sin((Omega_p-omega_p)/2)/np.sin((Omega_p+omega_p)/2)

    P = np.array([-alpha,1.0])

    Q = np.array([1.0,-alpha])

    order = max(len(b),len(a))-1

    bt = substitute_rational_polynomial(b,P,Q,order)

    at = substitute_rational_polynomial(a,P,Q,order)

    bt = bt/at[0]

    at = at/at[0]

    bt = clean_coefficients(bt)

    at = clean_coefficients(at)

    return alpha,bt,at

# ============================================================
# LP -> HP
# ============================================================

def transform_lp_to_hp(b,a,Omega_p,omega_p):

    alpha = np.cos((Omega_p-omega_p)/2)/np.cos((Omega_p+omega_p)/2)

    P = np.array([alpha,-1.0])

    Q = np.array([1.0,-alpha])

    order = max(len(b),len(a))-1

    bt_raw = substitute_rational_polynomial(b,P,Q,order)

    at_raw = substitute_rational_polynomial(a,P,Q,order)

    bt = bt_raw[::-1]

    at = at_raw[::-1]

    bt = bt/at[0]

    at = at/at[0]

    bt = clean_coefficients(bt)

    at = clean_coefficients(at)

    return alpha,bt,at

# ============================================================
# LP -> BP PARAMETERS
# ============================================================

def bp_parameters(Omega_p,omega_p1,omega_p2):

    alpha = np.cos((omega_p1+omega_p2)/2)/np.cos((omega_p1-omega_p2)/2)

    R = (np.cos(omega_p2)-alpha)/np.sin(omega_p2)

    kappa = np.tan(Omega_p/2)/R

    alpha2 = (kappa+1)/(kappa-1)

    alpha1 = -(1+alpha2)*alpha

    omega0 = np.arccos(np.clip(alpha,-1.0,1.0))

    return alpha,kappa,alpha1,alpha2,omega0

# ============================================================
# LP -> BP
# ============================================================

def transform_lp_to_bp(b,a,Omega_p,omega_p1,omega_p2):

    alpha,kappa,alpha1,alpha2,omega0 = bp_parameters(Omega_p,omega_p1,omega_p2)

    P = np.array([-alpha2,-alpha1,-1.0])

    Q = np.array([1.0,alpha1,alpha2])

    order = max(len(b),len(a))-1

    bt = substitute_rational_polynomial(b,P,Q,order)

    at = substitute_rational_polynomial(a,P,Q,order)

    bt = bt/at[0]

    at = at/at[0]

    bt = clean_coefficients(bt)

    at = clean_coefficients(at)

    return alpha,kappa,alpha1,alpha2,omega0,bt,at

# ============================================================
# LP -> BS PARAMETERS
# ============================================================

def bs_parameters(Omega_p,omega_p1,omega_p2):

    alpha = np.cos((omega_p1+omega_p2)/2)/np.cos((omega_p2-omega_p1)/2)

    kappa = np.tan((omega_p2-omega_p1)/2)*np.tan(Omega_p/2)

    alpha1 = -2*alpha/(1+kappa)

    alpha2 = (1-kappa)/(1+kappa)

    omega0 = np.arccos(np.clip(alpha,-1.0,1.0))

    return alpha,kappa,alpha1,alpha2,omega0

# ============================================================
# LP -> BS
# ============================================================

def transform_lp_to_bs(b,a,Omega_p,omega_p1,omega_p2):

    alpha,kappa,alpha1,alpha2,omega0 = bs_parameters(Omega_p,omega_p1,omega_p2)

    P = np.array([alpha2,alpha1,1.0])

    Q = np.array([1.0,alpha1,alpha2])

    order = max(len(b),len(a))-1

    bt = substitute_rational_polynomial(b,P,Q,order)

    at = substitute_rational_polynomial(a,P,Q,order)

    bt = bt/at[0]

    at = at/at[0]

    bt = clean_coefficients(bt)

    at = clean_coefficients(at)

    return alpha,kappa,alpha1,alpha2,omega0,bt,at

# ============================================================
# FREQUENCY GRID
# ============================================================

omega_grid = np.linspace(0,np.pi,1600)

_,H_proto = signal.freqz(b,a,worN=omega_grid)

prototype_limit = 1.08*max(np.max(np.abs(H_proto)),1.0)

# ============================================================
# INITIAL TRANSFORMATION
# ============================================================

alpha_initial,bt_initial,at_initial = transform_lp_to_lp(b,a,Omega_p,wp_slider.value*np.pi)

_,H_initial = signal.freqz(bt_initial,at_initial,worN=omega_grid)

# ============================================================
# FIGURE
# ============================================================

fig = plt.figure(figsize=(9.0,4.5))

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

gs = fig.add_gridspec(1,2,wspace=0.28)

ax_proto = fig.add_subplot(gs[0,0])

ax_trans = fig.add_subplot(gs[0,1])

fig.subplots_adjust(left=0.08,right=0.98,top=0.90,bottom=0.27,wspace=0.28)

# ============================================================
# PROTOTYPE RESPONSE
# ============================================================

proto_line, = ax_proto.plot(omega_grid/np.pi,np.abs(H_proto),linewidth=1.5,label='Prototype LP')

proto_pass_line = ax_proto.axvline(Omega_p/np.pi,linestyle='--',linewidth=1.1,label=r'$\Omega_p$')

proto_stop_line = ax_proto.axvline(Omega_s/np.pi,linestyle=':',linewidth=1.1,label=r'$\Omega_s$')

ax_proto.set_xlim(0,1)

ax_proto.set_ylim(0,prototype_limit)

ax_proto.set_title('Prototype Low-Pass Response')

ax_proto.set_xlabel(r'Prototype frequency $\Omega/\pi$')

ax_proto.set_ylabel(r'$|H(e^{j\Omega})|$')

ax_proto.grid(True,linestyle=':',alpha=0.28)

ax_proto.legend(loc='upper center',bbox_to_anchor=(0.5,-0.22),ncol=3,frameon=True)

# ============================================================
# TRANSFORMED RESPONSE
# ============================================================

trans_line, = ax_trans.plot(omega_grid/np.pi,np.abs(H_initial),linewidth=1.5,label='Transformed LP')

freq_line1 = ax_trans.axvline(0.65,linestyle='--',linewidth=1.1,label=r'$\omega_p$')

freq_line2 = ax_trans.axvline(0.65,linestyle='--',linewidth=1.1,visible=False)

center_line = ax_trans.axvline(0.60,linestyle=':',linewidth=1.0,visible=False)

ax_trans.set_xlim(0,1)

ax_trans.set_ylim(0,prototype_limit)

ax_trans.set_title('Transformed Low-Pass Response')

ax_trans.set_xlabel(r'New frequency $\omega/\pi$')

ax_trans.set_ylabel(r'$|\widetilde{H}(e^{j\omega})|$')

ax_trans.grid(True,linestyle=':',alpha=0.28)

trans_legend = ax_trans.legend(loc='upper center',bbox_to_anchor=(0.5,-0.22),ncol=2,frameon=True)

# ============================================================
# CONTROL VISIBILITY
# ============================================================

def update_control_visibility():

    transformation = transformation_radio.value

    if transformation in ['LP → LP','LP → HP']:

        single_frequency_box.layout.display = 'flex'

        band_frequency_box.layout.display = 'none'

    else:

        single_frequency_box.layout.display = 'none'

        band_frequency_box.layout.display = 'flex'

# ============================================================
# UPDATE
# ============================================================

def update(change=None):

    global trans_legend

    transformation = transformation_radio.value

    update_control_visibility()

    # --------------------------------------------------------
    # LP -> LP
    # --------------------------------------------------------

    if transformation == 'LP → LP':

        omega_p = wp_slider.value*np.pi

        alpha,bt,at = transform_lp_to_lp(b,a,Omega_p,omega_p)

        title = 'Transformed Low-Pass Response'

        curve_label = 'Transformed LP'

        parameter_text = f"""
        Prototype cutoff:
        <b>Ω<sub>p</sub> = {Omega_p/np.pi:.3f}π</b>
        &nbsp;&nbsp;&nbsp;
        New cutoff:
        <b>ω<sub>p</sub> = {omega_p/np.pi:.3f}π</b>
        &nbsp;&nbsp;&nbsp;
        <b>α = {alpha:.6f}</b>

        <div class="tr-equation">
        z = (z̃ − α)/(1 − αz̃)
        </div>
        """

        freq_line1.set_xdata([omega_p/np.pi,omega_p/np.pi])

        freq_line1.set_label(r'$\omega_p$')

        freq_line1.set_visible(True)

        freq_line2.set_visible(False)

        center_line.set_visible(False)

    # --------------------------------------------------------
    # LP -> HP
    # --------------------------------------------------------

    elif transformation == 'LP → HP':

        omega_p = wp_slider.value*np.pi

        alpha,bt,at = transform_lp_to_hp(b,a,Omega_p,omega_p)

        title = 'Transformed High-Pass Response'

        curve_label = 'Transformed HP'

        parameter_text = f"""
        Prototype cutoff:
        <b>Ω<sub>p</sub> = {Omega_p/np.pi:.3f}π</b>
        &nbsp;&nbsp;&nbsp;
        New cutoff:
        <b>ω<sub>p</sub> = {omega_p/np.pi:.3f}π</b>
        &nbsp;&nbsp;&nbsp;
        <b>α = {alpha:.6f}</b>

        <div class="tr-equation">
        z = −(z̃ − α)/(1 − αz̃)
        </div>
        """

        freq_line1.set_xdata([omega_p/np.pi,omega_p/np.pi])

        freq_line1.set_label(r'$\omega_p$')

        freq_line1.set_visible(True)

        freq_line2.set_visible(False)

        center_line.set_visible(False)

    # --------------------------------------------------------
    # LP -> BP
    # --------------------------------------------------------

    elif transformation == 'LP → BP':

        omega_p1 = wp1_slider.value*np.pi

        omega_p2 = wp2_slider.value*np.pi

        if omega_p1 >= omega_p2:

            result_html.value = """
            <div class="tr-result">
            <div class="tr-result-title">Invalid band specification</div>
            The lower passband edge must satisfy <b>ω<sub>p1</sub> &lt; ω<sub>p2</sub></b>.
            </div>
            """

            return

        alpha,kappa,alpha1,alpha2,omega0,bt,at = transform_lp_to_bp(b,a,Omega_p,omega_p1,omega_p2)

        title = 'Transformed Band-Pass Response'

        curve_label = 'Transformed BP'

        parameter_text = f"""
        Prototype cutoff:
        <b>Ω<sub>p</sub> = {Omega_p/np.pi:.3f}π</b>
        &nbsp;&nbsp;&nbsp;
        New passband:
        <b>{omega_p1/np.pi:.3f}π ≤ ω ≤ {omega_p2/np.pi:.3f}π</b>

        <br><br>

        <b>α = {alpha:.6f}</b>
        &nbsp;&nbsp;&nbsp;
        <b>κ = {kappa:.6f}</b>
        &nbsp;&nbsp;&nbsp;
        <b>α₁ = {alpha1:.6f}</b>
        &nbsp;&nbsp;&nbsp;
        <b>α₂ = {alpha2:.6f}</b>

        <div class="tr-equation">
        z = −(z̃² + α₁z̃ + α₂)/(1 + α₁z̃ + α₂z̃²)
        </div>

        Band center:
        <b>ω₀ = {omega0/np.pi:.3f}π</b>
        """

        freq_line1.set_xdata([omega_p1/np.pi,omega_p1/np.pi])

        freq_line2.set_xdata([omega_p2/np.pi,omega_p2/np.pi])

        center_line.set_xdata([omega0/np.pi,omega0/np.pi])

        freq_line1.set_label(r'$\omega_{p1}$')

        freq_line2.set_label(r'$\omega_{p2}$')

        center_line.set_label(r'$\omega_0$')

        freq_line1.set_visible(True)

        freq_line2.set_visible(True)

        center_line.set_visible(True)

    # --------------------------------------------------------
    # LP -> BS
    # --------------------------------------------------------

    else:

        omega_p1 = wp1_slider.value*np.pi

        omega_p2 = wp2_slider.value*np.pi

        if omega_p1 >= omega_p2:

            result_html.value = """
            <div class="tr-result">
            <div class="tr-result-title">Invalid band specification</div>
            The lower stopband edge must satisfy <b>ω<sub>p1</sub> &lt; ω<sub>p2</sub></b>.
            </div>
            """

            return

        alpha,kappa,alpha1,alpha2,omega0,bt,at = transform_lp_to_bs(b,a,Omega_p,omega_p1,omega_p2)

        title = 'Transformed Band-Stop Response'

        curve_label = 'Transformed BS'

        parameter_text = f"""
        Prototype cutoff:
        <b>Ω<sub>p</sub> = {Omega_p/np.pi:.3f}π</b>
        &nbsp;&nbsp;&nbsp;
        New rejected band:
        <b>{omega_p1/np.pi:.3f}π ≤ ω ≤ {omega_p2/np.pi:.3f}π</b>

        <br><br>

        <b>α = {alpha:.6f}</b>
        &nbsp;&nbsp;&nbsp;
        <b>κ = {kappa:.6f}</b>
        &nbsp;&nbsp;&nbsp;
        <b>α₁ = {alpha1:.6f}</b>
        &nbsp;&nbsp;&nbsp;
        <b>α₂ = {alpha2:.6f}</b>

        <div class="tr-equation">
        z = (z̃² + α₁z̃ + α₂)/(1 + α₁z̃ + α₂z̃²)
        </div>

        Rejection center:
        <b>ω₀ = {omega0/np.pi:.3f}π</b>
        """

        freq_line1.set_xdata([omega_p1/np.pi,omega_p1/np.pi])

        freq_line2.set_xdata([omega_p2/np.pi,omega_p2/np.pi])

        center_line.set_xdata([omega0/np.pi,omega0/np.pi])

        freq_line1.set_label(r'$\omega_{p1}$')

        freq_line2.set_label(r'$\omega_{p2}$')

        center_line.set_label(r'$\omega_0$')

        freq_line1.set_visible(True)

        freq_line2.set_visible(True)

        center_line.set_visible(True)

    # --------------------------------------------------------
    # RESPONSE
    # --------------------------------------------------------

    _,H_trans = signal.freqz(bt,at,worN=omega_grid)

    trans_line.set_ydata(np.abs(H_trans))

    trans_line.set_label(curve_label)

    ax_trans.set_title(title)

    transformed_limit = 1.08*max(np.max(np.abs(H_trans)),1.0)

    ax_trans.set_ylim(0,max(prototype_limit,transformed_limit))

    # --------------------------------------------------------
    # COEFFICIENTS
    # --------------------------------------------------------

    b_text = ', '.join([f'{value:.6f}' for value in bt])

    a_text = ', '.join([f'{value:.6f}' for value in at])

    poles = np.roots(at)

    maximum_pole_radius = np.max(np.abs(poles))

    result_html.value = f"""
    <div class="tr-result">

    <div class="tr-result-title">
    Current {transformation} transformation
    </div>

    {parameter_text}

    Maximum transformed pole radius:
    <b>{maximum_pole_radius:.6f}</b>

    <br><br>

    <b>Transformed numerator:</b>
    [{b_text}]

    <br>

    <b>Transformed denominator:</b>
    [{a_text}]

    </div>
    """

    # --------------------------------------------------------
    # LEGEND
    # --------------------------------------------------------

    if trans_legend is not None:

        trans_legend.remove()

    visible_lines = [trans_line]

    if freq_line1.get_visible():

        visible_lines.append(freq_line1)

    if freq_line2.get_visible():

        visible_lines.append(freq_line2)

    if center_line.get_visible():

        visible_lines.append(center_line)

    labels = [line.get_label() for line in visible_lines]

    legend_columns = 2 if len(visible_lines) <= 2 else 4

    trans_legend = ax_trans.legend(visible_lines,labels,loc='upper center',bbox_to_anchor=(0.5,-0.22),ncol=legend_columns,frameon=True)

    fig.canvas.draw_idle()

# ============================================================
# OBSERVERS
# ============================================================

transformation_radio.observe(update,names='value')

wp_slider.observe(update,names='value')

wp1_slider.observe(update,names='value')

wp2_slider.observe(update,names='value')

# ============================================================
# INITIAL VISIBILITY
# ============================================================

update_control_visibility()

# ============================================================
# DISPLAY
# ============================================================

display(header_html)

display(radio_box)

display(single_frequency_box)

display(band_frequency_box)

display(result_html)

display(fig.canvas)

update()